# Baseline model and evaluates its performance on the validation dataset.
# Logistic Regression is used as the first baseline model.
# The baseline will later be compared with more advanced models.

In [74]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score

In [2]:
import os
print(os.getcwd())

/Users/kamakshiilapavuluri/avazu_ctr_prediction


In [41]:
X_train = pd.read_parquet("data/processed/X_train.parquet")
y_train = pd.read_parquet("data/processed/y_train.parquet")["click"]
# When we load parquet file result will be in Dataframe

In [7]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (28300276, 26)
y_train: (28300276,)


In [8]:
# Load Validation Data
X_validation = pd.read_parquet("data/processed/X_validation.parquet")

y_validation = pd.read_parquet("data/processed/y_validation.parquet")["click"]

In [9]:
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

X_validation: (6064345, 26)
y_validation: (6064345,)


In [10]:
# Check the Data
X_train.head()

,C1,banner_pos,device_type,device_conn_type,C15,C16,C18,C19,C21,day,...,device_model_freq,site_domain_freq,site_id_freq,C14_freq,app_domain_freq,C17_freq,C20_freq,site_category_freq,app_category_freq,time_period_freq
0,1005,0,1,2,320,50,0,35,79,21,...,0.000583,0.173300,0.173300,0.014732,0.695755,0.139321,0.466963,0.196165,0.667639,0.228839
1,1010,1,4,0,320,50,3,35,117,21,...,0.009346,0.355104,0.342038,0.004268,0.695755,0.004268,0.466963,0.391277,0.232274,0.228839
2,1005,0,1,0,320,50,0,35,79,21,...,0.004185,0.173300,0.173300,0.015525,0.695755,0.139321,0.466963,0.196165,0.667639,0.228839
3,1005,1,1,0,320,50,0,681,101,21,...,0.000113,0.355104,0.342038,0.001250,0.107414,0.001250,0.001297,0.391277,0.232274,0.228839
4,1005,0,1,0,320,50,0,431,117,21,...,0.004286,0.000183,0.000158,0.009289,0.695755,0.009289,0.039648,0.082635,0.667639,0.228839


In [42]:
X_train.dtypes

C1                      int64
banner_pos              int64
device_type             int64
device_conn_type        int64
C15                     int64
C16                     int64
C18                     int64
C19                     int64
C21                     int64
day                      int8
hour_of_day              int8
day_of_week              int8
is_weekend               int8
device_id_freq        float64
device_ip_freq        float64
app_id_freq           float64
device_model_freq     float64
site_domain_freq      float64
site_id_freq          float64
C14_freq              float64
app_domain_freq       float64
C17_freq              float64
C20_freq              float64
site_category_freq    float64
app_category_freq     float64
time_period_freq      float32
dtype: object

In [43]:
# Check Missing Values
X_train.isnull().sum()

C1                    0
banner_pos            0
device_type           0
device_conn_type      0
C15                   0
C16                   0
C18                   0
C19                   0
C21                   0
day                   0
hour_of_day           0
day_of_week           0
is_weekend            0
device_id_freq        0
device_ip_freq        0
app_id_freq           0
device_model_freq     0
site_domain_freq      0
site_id_freq          0
C14_freq              0
app_domain_freq       0
C17_freq              0
C20_freq              0
site_category_freq    0
app_category_freq     0
time_period_freq      0
dtype: int64

In [44]:
validation_missing = (X_validation.isnull().sum().sum())

print("Validation missing values:",validation_missing)

Validation missing values: 0


In [45]:
# Check Class Distribution
y_train.value_counts()

click
0    23346931
1     4953345
Name: count, dtype: int64

In [46]:
y_train.value_counts(normalize=True) * 100

click
0    82.497185
1    17.502815
Name: proportion, dtype: float64

# Create Smaller Samples for Baseline Model Training
The complete training dataset contains more than 28 million rows.
Training multiple models on the complete dataset requires significant memory and computation time.
For initial model experimentation, a smaller random sample of the training and validation data will be used.
The sample is taken only after the time-based train, validation, and test split.
Therefore, future data is not introduced into the training dataset.
The final selected model can later be trained using a larger dataset.

In [47]:
# Select Sample Size
train_sample_size = 500_000
validation_sample_size = 100_000

In [48]:
# Create Training Sample
train_sample_indices = X_train.sample(
    n=train_sample_size,
    random_state=42
).index

In [49]:
X_train_sample = X_train.loc[
    train_sample_indices
].copy()

In [50]:
# Create matching target:
y_train_sample = y_train.loc[
    train_sample_indices
].copy()

In [51]:
# Create Validation Sample
validation_sample_indices = X_validation.sample(
    n=validation_sample_size,
    random_state=42
).index

In [54]:
X_validation_sample = X_validation.loc[
    validation_sample_indices
].copy()

# Create validation target sample

y_validation_sample = y_validation.loc[
    validation_sample_indices
].copy()

In [55]:
# Check Sample Sizes
print(
    "Training sample:",
    X_train_sample.shape
)

print(
    "Validation sample:",
    X_validation_sample.shape
)

print(
    "Training target:",
    y_train_sample.shape
)

print(
    "Validation target:",
    y_validation_sample.shape
)

Training sample: (500000, 26)
Validation sample: (100000, 26)
Training target: (500000,)
Validation target: (100000,)


In [56]:
# Check Click Distribution
print(
    "Original training click rate:"
)

print(
    y_train.mean()
)

Original training click rate:
0.17502815166890953


In [57]:
print(
    "Sample training click rate:"
)

print(
    y_train_sample.mean()
)

Sample training click rate:
0.175556


In [58]:
# Validation
print(
    "Validation sample click rate:"
)

print(
    y_validation_sample.mean()
)

Validation sample click rate:
0.14787


In [59]:
# Convert Data to float32
X_train_sample = X_train_sample.astype(
    "float32"
)

X_validation_sample = X_validation_sample.astype(
    "float32"
)

In [60]:
# Check Missing Values
print(
    X_train_sample.isnull().sum().sum()
)

print(
    X_validation_sample.isnull().sum().sum()
)

0
0


In [61]:
# Create StandardScaler
scaler = StandardScaler()

In [62]:
X_train_scaled = scaler.fit_transform(
    X_train_sample
)

In [63]:
# Apply the same scaler to validation
X_validation_scaled = scaler.transform(
    X_validation_sample
)

In [64]:
# Check Scaled Shapes
print(
    "Scaled training shape:",
    X_train_scaled.shape
)

print(
    "Scaled validation shape:",
    X_validation_scaled.shape
)


Scaled training shape: (500000, 26)
Scaled validation shape: (100000, 26)


In [65]:
# Create Logistic Regression Model
# L-BFGS is an optimization algorithm used by Logistic Regression to find model coefficients that minimize the model’s loss.
baseline_model = LogisticRegression(
    max_iter=100,
    solver="lbfgs"
)

In [66]:
# Fit the Training Data
baseline_model.fit(
    X_train_scaled,
    y_train_sample
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [67]:
# Predict Click Probabilities
validation_probabilities = baseline_model.predict_proba(
    X_validation_scaled
)[:, 1]

In [68]:
validation_probabilities[:10]

array([0.10993993, 0.05421156, 0.20614812, 0.09556675, 0.33546856,
       0.11193952, 0.08446413, 0.191512  , 0.10552502, 0.0544211 ],
      dtype=float32)

In [69]:
# Calculate Log Loss
validation_log_loss = log_loss(
    y_validation_sample,
    validation_probabilities
)

print(
    "Validation Log Loss:",
    round(validation_log_loss, 5)
)

Validation Log Loss: 0.40169


In [70]:
# Calculate ROC-AUC
validation_roc_auc = roc_auc_score(
    y_validation_sample,
    validation_probabilities
)

print(
    "Validation ROC-AUC:",
    round(validation_roc_auc, 5)
)

Validation ROC-AUC: 0.65108


## Second Model: Gradient Boosted Trees

After establishing the Logistic Regression baseline, a Gradient Boosted Tree
model will be evaluated.

Logistic Regression mainly learns linear relationships between features and the
target.

CTR behavior may contain nonlinear relationships and feature interactions. For
example, click behavior may depend on a combination of device type, time of day,
application frequency, and banner position.

Gradient Boosting builds multiple decision trees sequentially. Each new tree
attempts to improve the errors made by the previous trees.

The Gradient Boosting model will be compared with the Logistic Regression
baseline using Log Loss and ROC-AUC.

In [75]:
# Create Gradient Boosting Model
gradient_boosting_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

In [76]:
# Train the Model
gradient_boosting_model.fit(
    X_train_sample,
    y_train_sample
)

,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",6
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values

In [77]:
# Predict Validation Probabilities
gb_validation_probabilities = gradient_boosting_model.predict_proba(
    X_validation_sample
)[:, 1]

In [78]:
# Calculate Log Loss
gb_validation_log_loss = log_loss(
    y_validation_sample,
    gb_validation_probabilities
)

print(
    "Gradient Boosting Validation Log Loss:",
    round(gb_validation_log_loss, 5)
)

Gradient Boosting Validation Log Loss: 0.37606


In [79]:
# Calculate ROC-AUC
gb_validation_roc_auc = roc_auc_score(
    y_validation_sample,
    gb_validation_probabilities
)

print(
    "Gradient Boosting Validation ROC-AUC:",
    round(gb_validation_roc_auc, 5)
)

Gradient Boosting Validation ROC-AUC: 0.7308


In [80]:
# Compare with Logistic Regression
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "HistGradientBoosting"
    ],
    "Log_Loss": [
        validation_log_loss,
        gb_validation_log_loss
    ],
    "ROC_AUC": [
        validation_roc_auc,
        gb_validation_roc_auc
    ]
})

model_results

,Model,Log_Loss,ROC_AUC
0,Logistic Regression,0.401693,0.651082
1,HistGradientBoosting,0.376060,0.730798


# Gradient Boosting is currently the best-performing model inbetween these two the models.

# PR-AUC helps me evaluate how well the model identifies the click class when clicks are relatively rare.

In [82]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(
    y_validation_sample,
    gb_validation_probabilities
)

print(
    "PR-AUC:",
    round(pr_auc, 5)
)


PR-AUC: 0.32096


# Precision: Out of all records that predicted as clicks, how many actually clicked?

In [84]:
threshold = 0.5
gb_validation_predictions = (
    gb_validation_probabilities >= threshold
).astype(int)

In [86]:
from sklearn.metrics import precision_score

precision = precision_score(
    y_validation_sample,
    gb_validation_predictions
)

print(
    "Precision:",
    round(precision, 5)
)

Precision: 0.57878


# Recall Out of all actual clicks, how many did my model find?

In [87]:
from sklearn.metrics import recall_score

recall = recall_score(
    y_validation_sample,
    gb_validation_predictions
)

print(
    "Recall:",
    round(recall, 5)
)

Recall: 0.05018


# Calibration: Does the predicted probability match the real click rate?

In [88]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_validation_sample,
    gb_validation_probabilities,
    n_bins=10
)

In [89]:
print("Predicted probabilities:")
print(prob_pred)

print("Actual click rates:")
print(prob_true)

Predicted probabilities:
[0.06045637 0.14917845 0.24136347 0.32892589 0.44445294 0.54232044
 0.63595861 0.74219566]
Actual click rates:
[0.04281678 0.13352497 0.23688688 0.32449645 0.42440945 0.55791711
 0.57416268 0.73484848]


In [90]:
calibration_results = pd.DataFrame({
    "Predicted_Probability": prob_pred,
    "Actual_Click_Rate": prob_true
})

calibration_results

,Predicted_Probability,Actual_Click_Rate
0,0.060456,0.042817
1,0.149178,0.133525
2,0.241363,0.236887
3,0.328926,0.324496
4,0.444453,0.424409
5,0.542320,0.557917
6,0.635959,0.574163
7,0.742196,0.734848


In [91]:
calibration_results = pd.DataFrame({
    "Predicted_Probability": prob_pred,
    "Actual_Click_Rate": prob_true
})

calibration_results

,Predicted_Probability,Actual_Click_Rate
0,0.060456,0.042817
1,0.149178,0.133525
2,0.241363,0.236887
3,0.328926,0.324496
4,0.444453,0.424409
5,0.542320,0.557917
6,0.635959,0.574163
7,0.742196,0.734848


In [97]:
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

# Convert probabilities into click / no-click predictions
threshold = 0.5

predictions = (
    gb_validation_probabilities >= threshold
).astype(int)

# Calculate metrics
logloss = log_loss(
    y_validation_sample,
    gb_validation_probabilities
)

roc_auc = roc_auc_score(
    y_validation_sample,
    gb_validation_probabilities
)

pr_auc = average_precision_score(
    y_validation_sample,
    gb_validation_probabilities
)

precision = precision_score(
    y_validation_sample,
    predictions
)

recall = recall_score(
    y_validation_sample,
    predictions
)

# Show results
print("Log Loss:", round(logloss, 5))
print("ROC-AUC:", round(roc_auc, 5))
print("PR-AUC:", round(pr_auc, 5))
print("Precision:", round(precision, 5))
print("Recall:", round(recall, 5))

Log Loss: 0.37606
ROC-AUC: 0.7308
PR-AUC: 0.32096
Precision: 0.57878
Recall: 0.05018


## Evaluation Results

The HistGradientBoosting model achieved a Log Loss of 0.37606 and ROC-AUC of 0.7308.

The lower Log Loss shows that the model is producing better click probability predictions.

The ROC-AUC score shows that the model can reasonably separate click and non-click records.

The PR-AUC score of 0.32096 measures how well the model identifies the click class.

The precision of 0.57878 means that when the model predicts a click, about 58% of those predictions are correct.

The recall of 0.05018 means that, using a 0.5 probability threshold, the model identifies only about 5% of all actual clicks.

The low recall may be because the 0.5 threshold is high for CTR data. Different probability thresholds can be tested to improve the balance between precision and recall.